In [0]:
-- =====================================================================
-- Setup: schema + clean slate
-- =====================================================================
CREATE SCHEMA IF NOT EXISTS silver_demo;

DROP TABLE IF EXISTS silver_demo.order_items;
DROP TABLE IF EXISTS silver_demo.payments;
DROP TABLE IF EXISTS silver_demo.orders;
DROP TABLE IF EXISTS silver_demo.products;
DROP TABLE IF EXISTS silver_demo.customers;
DROP TABLE IF EXISTS silver_demo.accounts_clean;

-- =====================================================================
-- 1) Accounts (dimension)
-- =====================================================================
CREATE TABLE silver_demo.accounts_clean (
  AccountID   BIGINT,
  AccountName STRING,
  Country     STRING,
  AccountType STRING,
  IsActive    BOOLEAN
)
USING DELTA;

INSERT INTO silver_demo.accounts_clean (AccountID, AccountName, Country, AccountType, IsActive) VALUES
  (1, 'Walmart',   'USA',     'Retail',  true),
  (2, 'Target',    'USA',     'Retail',  true),
  (3, 'Loblaws',   'Canada',  'Retail',  true),
  (4, 'SportChek', 'Canada',  'Retail',  true),
  (5, 'Messer',    'Germany', 'Industrial', true),
  (6, 'Lidl',      'Germany', 'Retail',  true),
  (7, 'Mosmart',   'USA',     'Retail',  false),
  (8, 'Frazier',   'USA',     'Wholesale', true),
  (9, 'Shoppers',  'Canada',  'Pharmacy',  true),
  (10,'Hortons',   'Canada',  'Food',    true);

-- =====================================================================
-- 2) Customers (dimension, many-to-one to Accounts)
-- =====================================================================
CREATE TABLE silver_demo.customers (
  CustomerID BIGINT,
  AccountID  BIGINT,
  FirstName  STRING,
  LastName   STRING,
  Email      STRING,
  SignupDate DATE
)
USING DELTA;

INSERT INTO silver_demo.customers VALUES
  (101, 1,  'Ava',     'Reed',     'ava.reed@example.com',     DATE'2024-01-10'),
  (102, 1,  'Noah',    'Diaz',     'noah.diaz@example.com',    DATE'2024-02-02'),
  (103, 2,  'Maya',    'Singh',    'maya.singh@example.com',   DATE'2024-03-15'),
  (104, 3,  'Leo',     'Bennett',  'leo.bennett@example.com',  DATE'2024-04-20'),
  (105, 3,  'Sofia',   'Nguyen',   'sofia.nguyen@example.com', DATE'2024-04-28'),
  (106, 4,  'Ethan',   'Clark',    'ethan.clark@example.com',  DATE'2024-05-09'),
  (107, 6,  'Ivy',     'Klein',    'ivy.klein@example.com',    DATE'2024-06-01'),
  (108, 8,  'Omar',    'Lewis',    'omar.lewis@example.com',   DATE'2024-06-18'),
  (109, 9,  'Nora',    'Patel',    'nora.patel@example.com',   DATE'2024-07-05'),
  (110, 10, 'Liam',    'Martin',   'liam.martin@example.com',  DATE'2024-07-22');

-- =====================================================================
-- 3) Products (dimension)
-- =====================================================================
CREATE TABLE silver_demo.products (
  ProductID   BIGINT,
  SKU         STRING,
  ProductName STRING,
  Category    STRING,
  Price       DECIMAL(10,2)
)
USING DELTA;

INSERT INTO silver_demo.products VALUES
  (201, 'SKU-100', 'Trail Shoes',      'Footwear', 89.99),
  (202, 'SKU-110', 'City Sneakers',    'Footwear', 69.50),
  (203, 'SKU-200', 'Thermal Jacket',   'Apparel',  129.00),
  (204, 'SKU-210', 'Base Tee',         'Apparel',  19.99),
  (205, 'SKU-300', 'Protein Bars 12p', 'Grocery',  14.49),
  (206, 'SKU-400', 'Water Bottle 1L',  'Outdoors', 24.00);

-- =====================================================================
-- 4) Orders (header; many-to-one to Customers & Accounts)
-- =====================================================================
CREATE TABLE silver_demo.orders (
  OrderID    BIGINT,
  CustomerID BIGINT,
  AccountID  BIGINT,
  OrderDate  DATE,
  Status     STRING
)
USING DELTA;

INSERT INTO silver_demo.orders VALUES
  (1001, 101, 1,  DATE'2024-08-01', 'Shipped'),
  (1002, 101, 1,  DATE'2024-08-21', 'Pending'),
  (1003, 102, 1,  DATE'2024-09-05', 'Shipped'),
  (1004, 103, 2,  DATE'2024-09-10', 'Cancelled'),
  (1005, 104, 3,  DATE'2024-09-12', 'Shipped'),
  (1006, 105, 3,  DATE'2024-09-28', 'Shipped'),
  (1007, 106, 4,  DATE'2024-10-02', 'Pending'),
  (1008, 107, 6,  DATE'2024-10-05', 'Shipped'),
  (1009, 108, 8,  DATE'2024-10-09', 'Shipped'),
  (1010, 109, 9,  DATE'2024-10-10', 'Pending'),
  (1011, 110, 10, DATE'2024-10-11', 'Shipped'),
  (1012, 110, 10, DATE'2024-10-12', 'Pending');

-- =====================================================================
-- 5) Order items (detail; many-to-one to Orders & Products)
-- =====================================================================
CREATE TABLE silver_demo.order_items (
  OrderItemID BIGINT,
  OrderID     BIGINT,
  ProductID   BIGINT,
  Qty         INT,
  UnitPrice   DECIMAL(10,2)
)
USING DELTA;

INSERT INTO silver_demo.order_items VALUES
  (1, 1001, 201, 1,  89.99),
  (2, 1001, 206, 2,  24.00),
  (3, 1002, 204, 3,  19.99),
  (4, 1003, 203, 1, 129.00),
  (5, 1003, 205, 4,  14.49),
  (6, 1004, 202, 1,  69.50),      -- cancelled order
  (7, 1005, 203, 2, 129.00),
  (8, 1006, 201, 1,  89.99),
  (9, 1007, 204, 2,  19.99),
  (10,1008, 206, 1,  24.00),
  (11,1009, 205, 6,  14.49),
  (12,1010, 202, 1,  69.50),
  (13,1011, 201, 1,  89.99),
  (14,1011, 204, 2,  19.99),
  (15,1012, 206, 3,  24.00);

-- =====================================================================
-- 6) Payments (header-level; many-to-one to Orders)
-- =====================================================================
CREATE TABLE silver_demo.payments (
  PaymentID BIGINT,
  OrderID   BIGINT,
  PaidAt    TIMESTAMP,
  Amount    DECIMAL(12,2),
  Method    STRING,
  Success   BOOLEAN
)
USING DELTA;

INSERT INTO silver_demo.payments VALUES
  (5001, 1001, TIMESTAMP'2024-08-01 10:30:00', 137.99, 'Card',  true),
  (5002, 1003, TIMESTAMP'2024-09-05 15:12:00', 186.96, 'Card',  true),
  (5003, 1004, TIMESTAMP'2024-09-10 09:00:00',  69.50, 'Card',  false), -- cancelled/failed
  (5004, 1005, TIMESTAMP'2024-09-12 11:25:00', 258.00, 'ACH',   true),
  (5005, 1006, TIMESTAMP'2024-09-28 14:02:00',  89.99, 'Card',  true),
  (5006, 1008, TIMESTAMP'2024-10-05 16:45:00',  24.00, 'Card',  true),
  (5007, 1009, TIMESTAMP'2024-10-09 12:10:00',  86.94, 'Card',  true),
  (5008, 1011, TIMESTAMP'2024-10-11 18:33:00', 129.97, 'Card',  true);

# Creating Joins (inner, left, right, full outer)

In [0]:
-- 1) Orders with customers and accounts (classic inner joins)

SELECT
  o.OrderID, o.OrderDate, o.Status,
  c.CustomerID, c.FirstName, c.LastName,
  a.AccountName, a.Country
FROM silver_demo.orders o
JOIN silver_demo.customers c ON o.CustomerID = c.CustomerID
JOIN silver_demo.accounts_clean a ON o.AccountID = a.AccountID
ORDER BY o.OrderID;


In [0]:
-- now the same inner join but the results will be stored in a newly created table

CREATE OR REPLACE TABLE silver_demo.join_orders_customers AS
SELECT
  o.OrderID, o.OrderDate, o.Status,
  c.CustomerID, c.FirstName, c.LastName,
  a.AccountName, a.Country
FROM silver_demo.orders o
JOIN silver_demo.customers c ON o.CustomerID = c.CustomerID
JOIN silver_demo.accounts_clean a ON o.AccountID = a.AccountID
;

In [0]:
-- Now we do LEFT join (Accounts with or without orders)

SELECT
  a.AccountID, a.AccountName,
  COUNT(DISTINCT o.OrderID) AS orders_count
FROM silver_demo.accounts_clean a
LEFT JOIN silver_demo.orders o ON a.AccountID = o.AccountID
GROUP BY a.AccountID, a.AccountName
ORDER BY a.AccountID;

In [0]:
CREATE OR REPLACE TABLE silver_demo.join_accounts_orders AS
SELECT
  a.AccountID, a.AccountName,
  COUNT(DISTINCT o.OrderID) AS orders_count
FROM silver_demo.accounts_clean a
LEFT JOIN silver_demo.orders o ON a.AccountID = o.AccountID
GROUP BY a.AccountID, a.AccountName

In [0]:
-- Now a simple RIGHT JOIN (Orders with or without accounts)

CREATE OR REPLACE TABLE silver_demo.join_orders_accounts AS
SELECT
  a.AccountID,
  a.AccountName,
  o.OrderID,
  o.OrderDate
FROM silver_demo.orders o
RIGHT JOIN silver_demo.accounts_clean a
  ON o.AccountID = a.AccountID
ORDER BY a.AccountID;


In [0]:
-- Full outer join (see all rows including unmatxhed rows on both sides)

SELECT   a.AccountID AS a_AccountID, a.AccountName,   o.AccountID AS o_AccountID, o.OrderID FROM silver_demo.accounts_clean a FULL OUTER JOIN silver_demo.orders o ON a.AccountID = o.AccountID ORDER BY COALESCE(a.AccountID, o.AccountID), o.OrderID;

In [0]:
-- Full outer join (see all rows including unmatxhed rows on both sides)

SELECT   a.AccountID AS a_AccountID, a.AccountName,   o.AccountID AS o_AccountID, o.OrderID FROM silver_demo.accounts_clean a FULL OUTER JOIN silver_demo.orders o ON a.AccountID = o.AccountID ORDER BY COALESCE(a.AccountID, o.AccountID), o.OrderID;

In [0]:
-- Full outer join (see all rows including unmatxhed rows on both sides)

SELECT   a.AccountID AS a_AccountID, a.AccountName,   o.AccountID AS o_AccountID, o.OrderID FROM silver_demo.accounts_clean a FULL OUTER JOIN silver_demo.orders o ON a.AccountID = o.AccountID ORDER BY COALESCE(a.AccountID, o.AccountID), o.OrderID;

In [0]:
SELECT
  a.AccountID AS a_AccountID, a.AccountName,
  o.AccountID AS o_AccountID, o.OrderID
FROM silver_demo.accounts_clean a
FULL OUTER JOIN silver_demo.orders o
ON a.AccountID = o.AccountID
ORDER BY COALESCE(a.AccountID, o.AccountID), o.OrderID;

In [0]:

CREATE OR REPLACE TABLE silver_demo.outerjoin_orders_accounts AS
SELECT
  a.AccountID AS a_AccountID, a.AccountName,
  o.AccountID AS o_AccountID, o.OrderID
FROM silver_demo.accounts_clean a
FULL OUTER JOIN silver_demo.orders o
ON a.AccountID = o.AccountID
ORDER BY COALESCE(a.AccountID, o.AccountID), o.OrderID;